# Using the Crossref REST API

## 🎓 About Crossref

[Crossref](https://www.crossref.org/) is the largest **DOI registration agency** for scholarly content. Its members - publishers, universities, societies, funders - register **Digital Object Identifiers (DOIs)** for their publications and deposit the corresponding **metadata** with Crossref: titles, authors, ORCIDs, affiliations, references, funding information, licenses, links to full texts, corrections and retractions and much more. Today Crossref holds metadata on more than 180 million records: journal articles, books and chapters, conference papers, preprints, datasets, dissertations, reports, standards, peer reviews and even research grants.

Crossref makes this metadata **openly available** via its [REST API](https://www.crossref.org/documentation/retrieve-metadata/rest-api/) - free of charge and without registration. Crossref regards bibliographic metadata as facts which can be reused without restriction (please note that abstracts may remain subject to the publishers' copyright).

**💻 Endpoints**

Similar to other APIs for scholarly data, the Crossref REST API consists of different endpoints representing different kinds of entities:
* `/works` - the registered content items (articles, books, chapters, datasets ...) - the heart of the API
* `/funders` - funding organizations, based on the [Open Funder Registry](https://www.crossref.org/services/funder-registry/)
* `/members` - the Crossref members, i.e. the organizations that register DOIs (mostly publishers)
* `/journals` - journals and other serials, identified by their ISSN
* `/prefixes` - the DOI prefixes (like `10.1063`) owned by members
* `/types` - the content types used by Crossref
* `/licenses` - the license URLs attached to works

Most of these endpoints can be combined with `/works`, e.g. `/funders/{id}/works` lists all works funded by a certain funder. An interactive overview of all endpoints and parameters is available in the [Swagger documentation](https://api.crossref.org/swagger-ui/index.html).

**📦 Three types of responses**

* **Singleton**: the metadata record of a single object, e.g. `https://api.crossref.org/works/10.1063/1.1699114`
* **List**: the result of a query or filter, containing multiple items, e.g. `https://api.crossref.org/funders?query=swiss`
* **Headers only**: an HTTP `HEAD` request just tells you whether an object exists (status code 200) or not (404)

**☝️ Caveat**

All metadata in Crossref is **deposited by its members**. Its completeness and quality therefore varies a lot between publishers, journals and time periods: some deposit abstracts, references, affiliations and ORCIDs, others only the bare minimum. Citation counts in Crossref (`is-referenced-by-count`) only count citations from other Crossref records whose references were deposited. **Handle all derivative numbers with care** and crosscheck your findings with other data sources like [OpenAlex](https://openalex.org/), if possible. You will learn below how to check the metadata coverage of members and journals.

**😇 Be polite and show who you are**

Crossref offers three "pools" of API access:
* **Public pool**: anonymous requests
* **Polite pool**: requests which contain a contact e-mail address, either as `mailto` parameter or in the `User-Agent` header. They get **higher rate limits** and more reliable response times.
* **Plus pool**: for subscribers of the paid service [Metadata Plus](https://www.crossref.org/services/metadata-plus/), using an API token.

The rate limits were adjusted in 2025/2026 and are now differentiated by request type: requests for **single records** allow more requests per second than requests returning **lists** of records (queries, filters). The current limits are always given in the response headers `x-rate-limit-limit`, `x-rate-limit-interval` and `x-concurrency-limit` - we will look at them below. If you exceed the limits you will get the status code `429` (too many requests). So: **always provide your e-mail address**, send requests one after another (not in parallel) and pause shortly within loops.

In [1]:
# Check Python version
import sys
print(sys.version)

3.13.17 (main, Oct 22 2026, 14:06:04) [GCC 13.3.0]


In [2]:
# Install necessary libraries if not already available
#!pip install requests pandas habanero

In [3]:
# Import the necessary libraries
import json, os, re, time
import requests
import pandas as pd

In [4]:
# IMPORTANT: Fill in your e-mail address to use the polite pool
# (or set it as environment variable CROSSREF_MAILTO)
email = os.environ.get('CROSSREF_MAILTO', 'your.name@example.org')

We use a `requests.Session` for all requests: it reuses the connection to the server, which is faster, and allows us to set a descriptive `User-Agent` header once. Additionally, we pass our e-mail address as `mailto` parameter in every request.

We also tell the session to **automatically retry** a request a few times if the server answers with `429` (too many requests) or a server error (`5xx`), waiting a bit longer each time and respecting the `Retry-After` header of the server.

In [5]:
base_url = 'https://api.crossref.org/'

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()
session.headers.update({'User-Agent': f'ds-pytools Crossref tutorial (mailto:{email})'})

retry = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504], 
              allowed_methods=['GET', 'HEAD'], respect_retry_after_header=True)
session.mount('https://', HTTPAdapter(max_retries=retry))

## 📚 Works endpoint

The `/works` endpoint gives access to the metadata of all registered content items. A single *Work* is retrieved by its **DOI**: `https://api.crossref.org/works/{doi}`. DOIs are case-insensitive and may contain characters like `/`, `(` or `;`; the `requests` library takes care of encoding them correctly.

As a first example we retrieve the famous paper by Metropolis et al. (1953), which introduced the Metropolis algorithm.

In [6]:
doi = '10.1063/1.1699114'
#doi = '10.7554/elife.86663'               # open access article with license, full-text links and references
#doi = '10.1007/978-3-319-20319-5_4'       # book chapter

params = {'mailto': email}

r = session.get(base_url + 'works/' + doi, params=params)
print(r.status_code)
data = r.json()

200


Each response of the API is a JSON object with the same envelope: a `status`, the `message-type` (e.g. `work` or `work-list`), the `message-version` and the actual content in `message`.

In [7]:
# The response envelope
{key: data[key] for key in ['status', 'message-type', 'message-version']}

{'status': 'ok', 'message-type': 'work', 'message-version': '1.0.0'}

In [8]:
# The work's metadata is found in "message" - an overview of the available fields
work = data['message']
list(work)

['indexed',
 'reference-count',
 'publisher',
 'issue',
 'content-domain',
 'short-container-title',
 'published-print',
 'abstract',
 'DOI',
 'type',
 'created',
 'page',
 'update-policy',
 'source',
 'is-referenced-by-count',
 'title',
 'prefix',
 'volume',
 'author',
 'member',
 'reference',
 'container-title',
 'original-title',
 'language',
 'link',
 'deposited',
 'score',
 'resource',
 'subtitle',
 'short-title',
 'issued',
 'references-count',
 'journal-issue',
 'URL',
 'relation',
 'ISSN',
 'issn-type',
 'subject',
 'published-other',
 'published']

Note that the available fields **depend on what the publisher deposited**: a field which is missing in one record may be present in another. Therefore it is good practice to access fields with `.get()`, which returns `None` (or a given default) instead of raising an error. Also note that some fields like `title` or `container-title` are **lists**, since a work can have several titles (e.g. in different languages).

In [9]:
# Title, type, publisher and journal
print(work['title'])
print(work['type'])
print(work['publisher'])
print(work.get('container-title'), work.get('volume'), work.get('issue'), work.get('page'))

['Equation of State Calculations by Fast Computing Machines']
journal-article
AIP Publishing
['The Journal of Chemical Physics'] 21 6 1087-1092


In [10]:
# Authors with given and family name, sequence, affiliations and ORCID (if available)
work['author'][:2]

[{'given': 'Nicholas',
  'family': 'Metropolis',
  'sequence': 'first',
  'affiliation': [{'name': 'Los Alamos Scientific Laboratory, Los Alamos, New Mexico'}],
  'role': [{'vocabulary': 'crossref', 'role': 'author'}]},
 {'given': 'Arianna W.',
  'family': 'Rosenbluth',
  'sequence': 'additional',
  'affiliation': [{'name': 'Los Alamos Scientific Laboratory, Los Alamos, New Mexico'}],
  'role': [{'vocabulary': 'crossref', 'role': 'author'}]}]

In [11]:
# Print a readable author list
[f"{a.get('given', '')} {a.get('family', '')}" for a in work.get('author', [])]

['Nicholas Metropolis',
 'Arianna W. Rosenbluth',
 'Marshall N. Rosenbluth',
 'Augusta H. Teller',
 'Edward Teller']

Crossref records contain several **dates**, each given as `date-parts` (year, month, day - as far as known):
* `published-print` / `published-online`: the print and online publication dates
* `published`: the earliest of both
* `issued`: the earliest known publication date - the date to use for citations
* `created`, `deposited`, `indexed`: when the DOI was registered, the metadata was last deposited and indexed by Crossref

In [12]:
# Dates of the work
{key: work.get(key, {}).get('date-parts') for key in
 ['issued', 'published', 'published-print', 'published-online', 'created', 'deposited', 'indexed']}

{'issued': [[1953, 6, 1]],
 'published': [[1953, 6, 1]],
 'published-print': [[1953, 6, 1]],
 'published-online': None,
 'created': [[2005, 1, 5]],
 'deposited': [[2024, 2, 8]],
 'indexed': [[2026, 11, 5]]}

In [13]:
# Number of citations from other Crossref records and number of references of this work
print(work['is-referenced-by-count'])
print(work['references-count'])

31215
2


In [14]:
# The deposited references (if any) - some with DOI, some only as unstructured string
work.get('reference', [])[:2]

[{'key': '2024020814543804200_r1',
  'first-page': '1334',
  'volume': '20',
  'year': '1952',
  'journal-title': 'J. Chem. Phys.'},
 {'key': '2024020814543804200_r2'}]

In [15]:
# Member ID, DOI prefix and links to the work
print(work['member'], work['prefix'])
print(work['URL'])
print(work.get('resource'))

317 10.1063
https://doi.org/10.1063/1.1699114
{'primary': {'URL': 'https://pubs.aip.org/jcp/article/21/6/1087/202680/Equation-of-State-Calculations-by-Fast-Computing'}}


Further information is available for many works - but not for all: `abstract`, `license`, `link` (links to the full text, e.g. for text and data mining), `funder` (with award numbers), `relation` (to other works) and `update-to`/`updated-by` (corrections and retractions, see below).

**Abstracts** are delivered in [JATS XML](https://jats.nlm.nih.gov/) format. A simple regular expression is enough to remove the tags for a quick look.

In [16]:
# Abstract in JATS XML format - and as plain text
abstract = work.get('abstract', '')
print(abstract[:300])
print('---')
print(re.sub(r'<[^>]+>', ' ', abstract).split())

<jats:p>A general method, suitable for fast computing machines, for investigating such properties as equations of state for substances consisting of interacting individual molecules is described. The method consists of a modified Monte Carlo integration over configuration space. Results for the two-
---
['A', 'general', 'method,', 'suitable', 'for', 'fast', 'computing', 'machines,', 'for', 'investigating', 'such', 'properties', 'as', 'equations', 'of', 'state', 'for', 'substances', 'consisting', 'of', 'interacting', 'individual', 'molecules', 'is', 'described.', 'The', 'method', 'consists', 'of', 'a', 'modified', 'Monte', 'Carlo', 'integration', 'over', 'configuration', 'space.', 'Results', 'for', 'the', 'two-dimensional', 'rigid-sphere', 'system', 'have', 'been', 'obtained', 'on', 'the', 'Los', 'Alamos', 'MANIAC', 'and', 'are', 'presented', 'here.', 'These', 'results', 'are', 'compared', 'to', 'the', 'free', 'volume', 'equation', 'of', 'state', 'and', 'to', 'a', 'four-term', 'virial',

In [17]:
# Licenses and full-text links (here for an open access article from eLife)
r = session.get(base_url + 'works/10.7554/elife.86663', params=params)
elife = r.json()['message']

print(elife.get('license'))
elife.get('link')

[{'start': {'date-parts': [[2024, 1, 24]], 'date-time': '2024-01-24T00:00:00Z', 'timestamp': 1706054400000}, 'content-version': 'vor', 'delay-in-days': 0, 'URL': 'http://creativecommons.org/licenses/by/4.0/'}, {'start': {'date-parts': [[2024, 1, 24]], 'date-time': '2024-01-24T00:00:00Z', 'timestamp': 1706054400000}, 'content-version': 'am', 'delay-in-days': 0, 'URL': 'http://creativecommons.org/licenses/by/4.0/'}, {'start': {'date-parts': [[2024, 1, 24]], 'date-time': '2024-01-24T00:00:00Z', 'timestamp': 1706054400000}, 'content-version': 'tdm', 'delay-in-days': 0, 'URL': 'http://creativecommons.org/licenses/by/4.0/'}]


[{'URL': 'https://cdn.elifesciences.org/articles/86663/elife-86663-v1.pdf',
  'content-type': 'application/pdf',
  'content-version': 'vor',
  'intended-application': 'text-mining'},
 {'URL': 'https://cdn.elifesciences.org/articles/86663/elife-86663-v1.xml',
  'content-type': 'application/xml',
  'content-version': 'vor',
  'intended-application': 'text-mining'},
 {'URL': 'https://elifesciences.org/articles/86663',
  'content-type': 'unspecified',
  'content-version': 'vor',
  'intended-application': 'similarity-checking'}]

### Check existence and registration agency

Not every DOI is registered with Crossref: there are other registration agencies, especially [DataCite](https://datacite.org/), which registers DOIs for research data, software and repository content (e.g. Zenodo). If a DOI is unknown to Crossref, you will get the status code `404`.

To check whether a DOI exists **without downloading its metadata**, use an HTTP `HEAD` request. And the special route `/works/{doi}/agency` tells you **which agency registered a DOI**.

In [18]:
# HEAD request: 200 = exists in Crossref, 404 = unknown to Crossref
for d in ['10.1063/1.1699114', '10.5281/zenodo.3242074', '10.9999/does-not-exist']:
    r = session.head(base_url + 'works/' + d, params=params)
    print(d, r.status_code)

10.1063/1.1699114 200


10.5281/zenodo.3242074 404


10.9999/does-not-exist 404


In [19]:
# Find out the registration agency of a DOI
for d in ['10.1063/1.1699114', '10.5281/zenodo.3242074']:
    r = session.get(base_url + 'works/' + d + '/agency', params=params)
    print(d, r.json()['message']['agency'])

10.1063/1.1699114 {'id': 'crossref', 'label': 'Crossref'}


10.5281/zenodo.3242074 {'id': 'datacite', 'label': 'DataCite'}


### Get citations and references in other formats

Crossref can deliver the metadata of a single work in other formats: **BibTeX** or **RIS** for reference managers, **CSL JSON** for citation processors, or a **formatted citation** in one of the many [citation styles](https://citationstyles.org/). The general way to do this is [content negotiation](https://www.crossref.org/documentation/retrieve-metadata/content-negotiation/) at `https://doi.org/{doi}` with a corresponding `Accept` header, which works for Crossref and DataCite DOIs:

    requests.get('https://doi.org/10.1063/1.1699114', headers={'Accept': 'application/x-bibtex'})

The Crossref REST API offers the same functionality for Crossref DOIs with the route `/works/{doi}/transform/{format}`.

In [20]:
# BibTeX
r = session.get(base_url + 'works/' + doi + '/transform/application/x-bibtex', params=params)
r.encoding = 'utf-8'       # the server does not always declare the character encoding correctly
print(r.text)

 @article{Metropolis_1953, title={Equation of State Calculations by Fast Computing Machines}, volume={21}, ISSN={1089-7690}, url={http://dx.doi.org/10.1063/1.1699114}, DOI={10.1063/1.1699114}, number={6}, journal={The Journal of Chemical Physics}, publisher={AIP Publishing}, author={Metropolis, Nicholas and Rosenbluth, Arianna W. and Rosenbluth, Marshall N. and Teller, Augusta H. and Teller, Edward}, year={1953}, month=June, pages={1087–1092} }



In [21]:
# RIS
r = session.get(base_url + 'works/' + doi + '/transform/application/x-research-info-systems', params=params)
r.encoding = 'utf-8'
print(r.text)

TY  - JOUR
DO  - 10.1063/1.1699114
UR  - http://dx.doi.org/10.1063/1.1699114
TI  - Equation of State Calculations by Fast Computing Machines
T2  - The Journal of Chemical Physics
AU  - Metropolis, Nicholas
AU  - Rosenbluth, Arianna W.
AU  - Rosenbluth, Marshall N.
AU  - Teller, Augusta H.
AU  - Teller, Edward
PY  - 1953
DA  - 1953/06/01
PB  - AIP Publishing
SP  - 1087-1092
IS  - 6
VL  - 21
SN  - 0021-9606
SN  - 1089-7690
ER  - 



In [22]:
# Formatted citation: choose the citation style in the Accept header, e.g. apa, harvard-cite-them-right, ieee, nature
for style in ['apa', 'ieee', 'nature']:
    r = session.get(base_url + 'works/' + doi + '/transform',
                    params=params, headers={'Accept': f'text/x-bibliography; style={style}'})
    r.encoding = 'utf-8'
    print(style, ':', r.text)

apa : Metropolis, N., Rosenbluth, A. W., Rosenbluth, M. N., Teller, A. H., & Teller, E. (1953). Equation of State Calculations by Fast Computing Machines. The Journal of Chemical Physics, 21(6), 1087–1092. https://doi.org/10.1063/1.1699114



ieee : [1]N. Metropolis, A. W. Rosenbluth, M. N. Rosenbluth, A. H. Teller, and E. Teller, “Equation of State Calculations by Fast Computing Machines,” The Journal of Chemical Physics, vol. 21, no. 6, pp. 1087–1092, Jun. 1953, doi: 10.1063/1.1699114.



nature : 1.Metropolis, N., Rosenbluth, A. W., Rosenbluth, M. N., Teller, A. H. & Teller, E. Equation of State Calculations by Fast Computing Machines. The Journal of Chemical Physics 21, 1087–1092 (1953).



### Search works with queries

Instead of retrieving a single work, you can **search** the `/works` endpoint. The result is a **list** of works, sorted by **relevance** (`score`). There are two kinds of queries:

* `query` searches all fields of the metadata records
* **field queries** search only certain fields: `query.bibliographic` (all fields used for citations: title, authors, journal, year, volume, pages ...), `query.author`, `query.editor`, `query.affiliation`, `query.container-title`, `query.title`, `query.publisher-name`, `query.funder-name` and some more. They can be combined.

Please note: queries are **not** a precise boolean search - the terms are weighted and the results ranked by relevance. To select works exactly by certain criteria use **filters** (next section).

In [23]:
url = base_url + 'works'
params = {'mailto': email,
          'query': 'gauge field theory',
          'rows': 10}

r = session.get(url, params=params)
data = r.json()

In [24]:
# A list response contains some general information and the "items"
print(list(data['message']))
print('total results =', data['message']['total-results'])
print('items per page =', data['message']['items-per-page'])

['facets', 'total-results', 'items', 'items-per-page', 'query']
total results = 3242100
items per page = 10


In [25]:
# The 10 most relevant results with score, DOI and title
for item in data['message']['items']:
    print(round(item['score'], 1), item['DOI'], item.get('title'))

27.3 10.1201/9780203750100-9 ['Gauge Field Theories']
27.1 10.1017/9781108585286.010 ['Nonabelian Gauge Theories']
26.8 10.1887/0750302674/b552b5 ['Index']
26.6 10.1887/0750302674/b552c8 ['The superstring']
26.6 10.1017/cbo9780511524493.007 ['Gauge theory']
26.5 10.1887/0750302674/b552c4 ['Pure supergravity']
26.5 10.1887/0750302674/b552c12 ['Superstring interactions']
26.3 10.1007/978-3-642-22421-8_14 ['The Commutative Weyl U(1)-Gauge Theory and the Electromagnetic Field']
26.3 10.1017/cbo9780511813917.080 ['Background field gauge (73)']
26.2 10.1093/oso/9780198861492.003.0005 ['Classical Gauge Field Theory']


In [26]:
# Combine field queries: works by an author named "Thiel" with "Mülheim" in the affiliation
params = {'mailto': email,
          'query.author': 'Walter Thiel',
          'query.affiliation': 'Mülheim',
          'rows': 5}

r = session.get(url, params=params)
data = r.json()
print('total results =', data['message']['total-results'])

for item in data['message']['items']:
    print(item['DOI'], item.get('title'), [a.get('family') for a in item.get('author', [])])

total results = 475
10.1002/wcms.1161 ['Semiempirical quantum–chemical methods'] ['Thiel']
10.1021/jp053883v ['Thermochemistry of the Fluoroformyloxyl Radical:\u2009 A Computational Study Based on Coupled Cluster Theory'] ['Breidung', 'Thiel']
10.1021/ct049844p ['Enzymatic Hydroxylation in <i>p</i>-Hydroxybenzoate Hydroxylase:\u2009 A Case Study for QM/MM Molecular Dynamics'] ['Senn', 'Thiel', 'Thiel']
10.1021/jp506101x ['Photoinduced Intramolecular Charge Transfer in an Electronically Modified Flavin Derivative: Roseoflavin'] ['Karasulu', 'Thiel']
10.1021/jp0685986 ['Substrate Orientation in 4-Oxalocrotonate Tautomerase and Its Effect on QM/MM Energy Profiles'] ['Tuttle', 'Thiel']


#### Match references with `query.bibliographic`

A very useful application of `query.bibliographic` is **reference matching**: finding the DOI for a reference given as plain text, e.g. from a reference list. Look at the `score` and compare the top results - the first hit is not always the work you are looking for! In the example below the first result is a technical report with the same title, published by the same authors before the journal article.

In [27]:
reference = 'Metropolis N, Rosenbluth AW, Rosenbluth MN, Teller AH, Teller E (1953) Equation of state calculations by fast computing machines. J Chem Phys 21:1087'

params = {'mailto': email,
          'query.bibliographic': reference,
          'rows': 3}

r = session.get(url, params=params)
for item in r.json()['message']['items']:
    print(round(item['score'], 1), item['DOI'], item['type'], item.get('container-title'), item.get('title'))

126.8 10.2172/4390578 report None ['Equation of state calculations by fast computing machines']
120.6 10.1063/1.1699114 journal-article ['The Journal of Chemical Physics'] ['Equation of State Calculations by Fast Computing Machines']
95.0 10.1007/978-1-4612-0667-5_6 book-chapter ['Springer Series in Statistics', 'Breakthroughs in Statistics'] ['Introduction to Metropolis, Rosenbluth, Rosenbluth, Teller, and Teller (1953) Equations of State Calculations by Fast Computing Machines. J. Chem. Phys.,21, 1087–1092. and Geman and Geman (1984) Stochastic Relaxation, Gibbs Distributions, and the Bayesian Restoration of Images. IEEE Trans. Pattern Anal. Machine Intelligence,6, 721–741.']


### Filter works

**Filters** select works exactly by certain criteria. They are given in the `filter` parameter as `name:value` pairs, separated by commas. There are more than 100 filters for works, e.g.:

| Filter | Description |
|---|---|
| `type:journal-article` | content type (see `/types` endpoint below) |
| `from-pub-date:2024-01-01`, `until-pub-date:2024-12-31` | publication date range (also `-created-`, `-deposit-`, `-update-`, `-index-`, `-online-pub-`, `-print-pub-` dates) |
| `funder:10.13039/501100001711` | funded by a certain funder (Funder ID) |
| `orcid:0000-0001-6780-0350` | contributions of an author (ORCID) |
| `ror-id:02k7v4d05` | affiliation with an institution ([ROR](https://ror.org/) ID) |
| `issn:0028-0836`, `member:311`, `prefix:10.1063` | journal, member, DOI prefix |
| `has-abstract:true`, `has-references:true`, `has-orcid:true`, `has-funder:true`, `has-license:true`, `has-full-text:true` | availability of certain metadata |
| `license.url:...`, `license.delay:0` | license URL, embargo in days |
| `update-type:retraction`, `updates:{doi}`, `is-update:true` | corrections, retractions etc. |
| `relation.type:is-preprint-of` | relations between works |
| `award.number:...`, `clinical-trial-number:...` | grant and clinical trial numbers |

The complete list is in the [filter documentation](https://www.crossref.org/documentation/retrieve-metadata/rest-api/rest-api-filters/). A useful trick: if you use an invalid filter name, the error message lists all valid filters (see "Error handling" at the end).

Different filters are combined with **AND**; the same filter repeated with different values is combined with **OR**.

In [28]:
# Journal articles funded by the Swiss National Science Foundation (SNSF), published in 2025
params = {'mailto': email,
          'filter': 'funder:10.13039/501100001711,type:journal-article,from-pub-date:2025-01-01,until-pub-date:2025-12-31',
          'rows': 5}

r = session.get(url, params=params)
data = r.json()
print('total results =', data['message']['total-results'])

for item in data['message']['items']:
    print(item['DOI'], item.get('container-title'), item.get('title'))

total results = 7343
10.1007/s10107-024-02177-x ['Mathematical Programming'] ['Constant-competitiveness for random assignment Matroid secretary without knowing the Matroid']
10.1051/sm/2025023 ['Movement &amp; Sport Sciences - Science &amp; Motricité'] ['Développement moteur et activité physique dans l’enfance\u2009–\u2009relation entre les capacités motrices de base et les activités de loisirs des enfants en Suisse romande']
10.3390/nu17060988 ['Nutrients'] ['Programming the Brain: How Maternal Overnutrition Shapes Cognitive Aging in Offspring']
10.1016/j.jrtpm.2025.100555 ['Journal of Rail Transport Planning &amp; Management'] ['Optimizing periodic stability in railway timetables: A microscopic model for networks with a macroscopic comparison']
10.1016/j.jtha.2025.03.021 ['Journal of Thrombosis and Haemostasis'] ['Enhancing hemostasis potency in hemophilia with a small interfering ribonucleic acid targeting protein S']


In [29]:
# The same filter with different values means OR: books or monographs
for f in ['type:book', 'type:monograph', 'type:book,type:monograph']:
    r = session.get(url, params={'mailto': email, 'filter': f, 'rows': 0})
    print(f, r.json()['message']['total-results'])

type:book 858403
type:monograph 819172


type:book,type:monograph 1677575


Note the parameter `rows=0` in the last cell: it returns **no items, only the number of results** - a fast and cheap way to count.

Filters and queries can be combined, of course. In the next example we look for works of a certain author identified by ORCID and sort them by the number of citations, using the parameters `sort` and `order`. Possible sort fields are `relevance` (default for queries), `score`, `published`, `issued`, `created`, `updated`, `deposited`, `indexed`, `is-referenced-by-count` and `references-count`.

In [30]:
# The most cited works of Walter Thiel (ORCID 0000-0001-6780-0350)
params = {'mailto': email,
          'filter': 'orcid:0000-0001-6780-0350',
          'sort': 'is-referenced-by-count',
          'order': 'desc',
          'rows': 5}

r = session.get(url, params=params)
data = r.json()
print('total results =', data['message']['total-results'])

for item in data['message']['items']:
    print(item['is-referenced-by-count'], item['issued']['date-parts'], item.get('container-title'), item.get('title'))

total results = 41
393 [[2017, 1, 3]] ['Journal of Chemical Theory and Computation'] ['Evaluation of Spin-Orbit Couplings with Linear-Response Time-Dependent Density Functional Methods']
163 [[2018, 9, 10]] ['The Journal of Physical Chemistry Letters'] ['Nonadiabatic Excited-State Dynamics with Machine Learning']
155 [[2017, 2, 7]] ['Journal of the American Chemical Society'] ['Ruthenium-Catalyzed Alkyne <i>trans</i>-Hydrometalation: Mechanistic Insights and Preparative Implications']
151 [[2018, 2, 16]] ['Journal of the American Chemical Society'] ['Half-Sandwich Ruthenium Carbene Complexes Link\n                    <i>trans</i>\n                    -Hydrogenation and\n                    <i>gem</i>\n                    -Hydrogenation of Internal Alkynes']
149 [[2017, 6, 27]] ['The Journal of Chemical Physics'] ['Structure-based sampling and self-correcting machine learning for accurate calculations of potential energy surfaces and vibrational levels']


#### Select fields and get random samples

Full metadata records can be large - especially due to reference lists. With the `select` parameter you can choose the fields you need, which makes responses much smaller and faster. The `sample` parameter returns a **random sample** of up to 100 works - useful for statistical analyses of metadata quality.

In [31]:
# A random sample of 5 works with only some selected fields
params = {'mailto': email,
          'sample': 5,
          'select': 'DOI,type,issued,title,publisher'}

r = session.get(url, params=params)
r.json()['message']['items']

[{'DOI': '10.1007/bf00659951',
  'type': 'journal-article',
  'issued': {'date-parts': [[1989, 8]]},
  'title': ['Fluorescence spectra of different configurations of the diphenyl molecule in solid paraffin solutions'],
  'publisher': 'Springer Science and Business Media LLC'},
 {'DOI': '10.1016/s0003-4975(00)01505-8',
  'type': 'journal-article',
  'issued': {'date-parts': [[2000, 8]]},
  'title': ['Effects of minimal invasive coronary artery bypass on pulmonary function and postoperative pain'],
  'publisher': 'Elsevier BV'},
 {'DOI': '10.47749/t/unicamp.2013.917842',
  'type': 'dissertation',
  'issued': {'date-parts': [[None]]},
  'title': ['Masticatory function in partially edentulous patients after rehabilitation with removable and fixed prosthesis over osseointegrated implants'],
  'publisher': 'Universidade Estadual de Campinas'},
 {'DOI': '10.1201/9780203021941-24',
  'type': 'book-chapter',
  'issued': {'date-parts': [[2004, 8, 30]]},
  'title': ['Autoimmune Type 1 Diabetes'],

In [32]:
# Combine filter, sample and select: a random sample of 20 journal articles published in 2025.
# How many of them have an abstract, references or ORCIDs?
params = {'mailto': email,
          'filter': 'type:journal-article,from-pub-date:2025-01-01,until-pub-date:2025-12-31',
          'sample': 20,
          'select': 'DOI,abstract,reference,author'}

r = session.get(url, params=params)
items = r.json()['message']['items']

print('with abstract:  ', sum('abstract' in i for i in items))
print('with references:', sum('reference' in i for i in items))
print('with ORCIDs:    ', sum(any('ORCID' in a for a in i.get('author', [])) for i in items))

with abstract:   6
with references: 14
with ORCIDs:     9


### Facets: summary statistics

**Facets** give you summary statistics of a result set without downloading the records - similar to the `group_by` parameter of OpenAlex. Use the parameter `facet` with the name of the facet and the maximum number of values (up to 1000, `*` for the maximum). Available facets are e.g. `type-name`, `published` (year), `publisher-name`, `container-title`, `funder-name`, `affiliation`, `ror-id`, `license`, `orcid`, `issn`, `update-type`, `relation-type` and `category-name`. Several facets can be requested at once, separated by commas.

Note that facet counts are approximate. Combined with `rows=0`, facets are a very efficient tool for bibliometric overviews.

In [33]:
# All works in Crossref by content type
params = {'mailto': email,
          'facet': 'type-name:*',
          'rows': 0}

r = session.get(url, params=params)
data = r.json()
print('total results =', data['message']['total-results'])
data['message']['facets']['type-name']['values']

total results = 188482738


{'Journal Article': 125029595,
 'Chapter': 24915863,
 'Conference Paper': 10218298,
 'Component': 9484719,
 'Posted Content': 4171959,
 'Dataset': 3478518,
 'Other': 1948263,
 'Journal Issue': 1513783,
 'Peer Review': 1150321,
 'Dissertation': 1101481,
 'Report': 979193,
 'Entry': 932165,
 'Book': 858403,
 'Monograph': 819172,
 'Edited Book': 592047,
 'Standard': 419054,
 'Grant': 245848,
 'Reference': 216279,
 'Proceedings': 86007,
 'Journal': 84402,
 'Report Component': 69837,
 'Section': 53977,
 'Database': 44171,
 'Part': 28463,
 'Journal Volume': 13469,
 'Report Series': 5388,
 'Book Series': 3563,
 'Book Set': 662,
 'Proceedings Series': 380,
 'Track': 52}

In the next example we combine a **filter** for works affiliated with the *University of Bern* (ROR ID `02k7v4d05`) with several facets. Please note that only works with ROR IDs in their affiliation metadata are found - a quite recent and not yet common practice of publishers. So this is by far not the complete output of the University of Bern!

In [34]:
params = {'mailto': email,
          'filter': 'ror-id:02k7v4d05',
          'facet': 'published:*,publisher-name:10,type-name:*',
          'rows': 0}

r = session.get(url, params=params)
data = r.json()
print('total results =', data['message']['total-results'])
facets = data['message']['facets']

total results = 1136


In [35]:
# Publications per year, sorted by year
sorted(facets['published']['values'].items(), reverse=True)[:10]

[('2026', 402),
 ('2025', 319),
 ('2024', 224),
 ('2023', 61),
 ('2022', 58),
 ('2021', 30),
 ('2020', 4),
 ('2019', 7),
 ('2018', 1),
 ('2017', 2)]

In [36]:
# The 10 most frequent publishers and the content types
print(facets['publisher-name']['values'])
print(facets['type-name']['values'])

{'Springer Science and Business Media LLC': 288, 'Copernicus GmbH': 175, 'eLife Sciences Publications, Ltd': 102, 'American Chemical Society (ACS)': 102, 'American Physical Society (APS)': 79, 'Front Matter': 55, 'John Benjamins Publishing Company': 51, 'Cambridge University Press (CUP)': 37, 'American Society for Microbiology': 32, 'SJORANM GmbH': 22}
{'Journal Article': 776, 'Posted Content': 267, 'Chapter': 38, 'Peer Review': 27, 'Grant': 7, 'Part': 5, 'Conference Paper': 5, 'Edited Book': 4, 'Report': 3, 'Monograph': 2, 'Other': 1, 'Book': 1}


### Retrieve large result sets with cursors

A single request returns at most **1000 items** (`rows`). To get more results, there are two options:
* `offset`: skip the first *n* results - limited to 10,000 results in total
* `cursor`: **deep paging** without limits. Start with `cursor=*`; every response contains a `next-cursor` value which you pass on in the next request. Stop when a page contains fewer items than requested. Cursors expire after 5 minutes.

Crossref recommends using cursors. In the example we retrieve all works affiliated with the University of Bern (via ROR ID) published since 2024, selecting only a few fields, and put them into a pandas dataframe. Note the short pause within the loop to respect the rate limits.

In [37]:
results = []
cursor = '*'
rows = 200

while True:
    params = {'mailto': email,
              'filter': 'ror-id:02k7v4d05,from-pub-date:2024-01-01',
              'select': 'DOI,type,issued,title,container-title,publisher,is-referenced-by-count',
              'rows': rows,
              'cursor': cursor}
    r = session.get(url, params=params)
    message = r.json()['message']
    results.extend(message['items'])
    print(len(results), 'of', message['total-results'])
    if len(message['items']) < rows:
        break
    cursor = message['next-cursor']
    time.sleep(0.5)

200 of 944


400 of 944


600 of 944


800 of 944


944 of 944


In [38]:
# Transform the selected fields into a dataframe
df = pd.DataFrame({
    'doi': [i['DOI'] for i in results],
    'type': [i['type'] for i in results],
    'year': [i['issued']['date-parts'][0][0] for i in results],
    'title': [i.get('title', [None])[0] for i in results],
    'container': [(i.get('container-title') or [None])[0] for i in results],
    'publisher': [i.get('publisher') for i in results],
    'cited_by': [i.get('is-referenced-by-count') for i in results]})
df.head(10)

,doi,type,year,title,container,publisher,cited_by
0,10.7554/elife.86663,journal-article,2024,Dally is not essential for Dpp spreading or in...,eLife,"eLife Sciences Publications, Ltd",2
1,10.7554/elife.89601,journal-article,2024,Episodic long-term memory formation during slo...,eLife,"eLife Sciences Publications, Ltd",6
2,10.7554/elife.89674,journal-article,2024,A neuronal least-action principle for real-tim...,eLife,"eLife Sciences Publications, Ltd",12
3,10.7554/elife.89855,journal-article,2024,High-resolution kinetics of herbivore-induced ...,eLife,"eLife Sciences Publications, Ltd",9
4,10.7554/elife.90502,journal-article,2024,Transformer-based spatial–temporal detection o...,eLife,"eLife Sciences Publications, Ltd",5
5,10.7554/elife.90305,journal-article,2026,GTPase-activating protein DLC1 spatio-temporal...,eLife,"eLife Sciences Publications, Ltd",4
6,10.1107/s2052520623010636,journal-article,2024,Lattice response to the radiation damage of mo...,Acta Crystallographica Section B Structural Sc...,International Union of Crystallography (IUCr),8
7,10.1007/jhep01(2024)004,journal-article,2024,Differential cross-section measurements of the...,Journal of High Energy Physics,Springer Science and Business Media LLC,9
8,10.7554/elife.94075,journal-article,2024,Germline cis variant determines epigenetic reg...,eLife,"eLife Sciences Publications, Ltd",3
9,10.1007/jhep01(2024)066,journal-article,2024,Studies of new Higgs boson interactions throug...,Journal of High Energy Physics,Springer Science and Business Media LLC,14


In [39]:
# Number of works per year and type
df.groupby(['year', 'type']).size()

year  type               
2024  book-chapter             1
      journal-article        151
      posted-content          69
      proceedings-article      2
      report                   1
2025  book-chapter            14
      edited-book              1
      journal-article        238
      monograph                1
      other                    1
      peer-review              2
      posted-content          61
2026  book                     1
      book-chapter             1
      book-part                2
      edited-book              1
      journal-article        312
      monograph                1
      peer-review              6
      posted-content          74
      proceedings-article      3
      report                   1
dtype: int64

In [40]:
# The 5 most cited works in this set
df.sort_values('cited_by', ascending=False).head(5)

,doi,type,year,title,container,publisher,cited_by
115,10.1103/prxquantum.5.037001,journal-article,2024,Quantum Computing for High-Energy Physics: Sta...,PRX Quantum,American Physical Society (APS),235
333,10.1103/nfzp-p5dn,journal-article,2026,FLAG review 2024,Physical Review D,American Physical Society (APS),103
236,10.1140/epjc/s10052-024-13701-w,journal-article,2025,Software and computing for Run 3 of the ATLAS ...,The European Physical Journal C,Springer Science and Business Media LLC,64
524,10.1140/epjc/s10052-025-15077-x,journal-article,2025,Future Circular Collider Feasibility Study Report,The European Physical Journal C,Springer Science and Business Media LLC,59
35,10.1007/s10792-024-03057-1,journal-article,2024,Corneal biomechanics and diagnostics: a review﻿,International Ophthalmology,Springer Science and Business Media LLC,41


### Corrections, retractions and other updates

Publications can be **updated** after publication by corrections, errata, retractions, expressions of concern etc. Crossref records these updates in two directions:
* the **update notice** (e.g. a retraction notice) has the field `update-to`, pointing to the original work
* the **updated work** has the field `updated-by`, pointing to the notice

Crossref acquired the [Retraction Watch](https://retractionwatch.com/) database in 2023 and integrates its data into the API, so retractions are covered even if the publisher did not deposit them. The `source` attribute tells you whether an update was asserted by the `publisher` or by `retraction-watch`. This is especially valuable to **check reference lists for retracted works**.

In [41]:
# How many updates of which kind are there?
params = {'mailto': email,
          'facet': 'update-type:10',
          'rows': 0}

r = session.get(url, params=params)
r.json()['message']['facets']['update-type']['values']

{'correction': 217760,
 'erratum': 118634,
 'retraction': 81358,
 'new_version': 45175,
 'new_edition': 10914,
 'corrigendum': 8890,
 'expression_of_concern': 4443,
 'withdrawal': 3420,
 'addendum': 1764,
 'removal': 708}

In [42]:
# A retracted work and its retraction notice
r = session.get(base_url + 'works/10.1177/1758835919874651', params={'mailto': email})
retracted = r.json()['message']

print(retracted['title'])
retracted.get('updated-by')

['RETRACTED: Downregulation of long noncoding RNA LINC01419 inhibits cell migration, invasion, and tumor growth and promotes autophagy\n                    <i>via</i>\n                    inactivation of the PI3K/Akt1/mTOR pathway in gastric cancer']


[{'DOI': '10.1177/17588359211061903',
  'type': 'retraction',
  'label': 'Retraction',
  'source': 'retraction-watch',
  'updated': {'date-parts': [[2021, 12, 15]],
   'date-time': '2021-12-15T00:00:00Z',
   'timestamp': 1639526400000},
  'record-id': '34999'},
 {'DOI': '10.1177/17588359211061903',
  'type': 'retraction',
  'label': 'Retraction',
  'source': 'publisher',
  'updated': {'date-parts': [[2021, 12, 15]],
   'date-time': '2021-12-15T00:00:00Z',
   'timestamp': 1639526400000}}]

In [43]:
# Find all update notices for a given DOI with the "updates" filter
params = {'mailto': email,
          'filter': 'updates:10.1177/1758835919874651'}

r = session.get(url, params=params)
for item in r.json()['message']['items']:
    print(item['DOI'], item.get('title'), [(u['type'], u['source']) for u in item['update-to']])

10.1177/17588359211061903 ['Retraction notice: Downregulation of long noncoding RNA LINC01419 inhibits cell migration, invasion, and tumor growth and promotes autophagy via inactivation of the PI3K/Akt1/mTOR pathway in gastric cancer.'] [('retraction', 'retraction-watch'), ('retraction', 'publisher')]


In [44]:
# A simple check: are there retracted works among a list of DOIs (e.g. from your reference list)?
dois = ['10.1063/1.1699114', '10.1177/1758835919874651', '10.7554/elife.86663']

for d in dois:
    r = session.get(base_url + 'works/' + d, params={'mailto': email})
    updates = [u['type'] for u in r.json()['message'].get('updated-by', [])]
    print(d, '- RETRACTED' if 'retraction' in updates else '- ok', updates)
    time.sleep(0.2)

10.1063/1.1699114 - ok []


10.1177/1758835919874651 - RETRACTED ['retraction', 'retraction']


10.7554/elife.86663 - ok []


### Relations between works

The field `relation` connects works with each other, e.g. a preprint with its published version (`is-preprint-of` / `has-preprint`), a peer review with the reviewed article (`is-review-of`), translations, versions, supplements or comments. Relations can also point to non-DOI identifiers. Let's look at the most frequent relation types and an example.

In [45]:
params = {'mailto': email,
          'facet': 'relation-type:15',
          'rows': 0}

r = session.get(url, params=params)
r.json()['message']['facets']['relation-type']['values']

{'is-component-of': 8680839,
 'is-review-of': 1402241,
 'is-preprint-of': 849470,
 'is-identical-to': 832630,
 'has-preprint': 585241,
 'has-review': 453131,
 'is-supplement-to': 235488,
 'is-part-of': 215239,
 'is-version-of': 172171,
 'has-version': 150020,
 'is-comment-on': 111337,
 'is-supplemented-by': 104767,
 'correction': 79334,
 'is-translation-of': 72524,
 'has-translation': 63510}

In [46]:
# Preprints with a link to their published version
params = {'mailto': email,
          'filter': 'relation.type:is-preprint-of,from-pub-date:2025-01-01',
          'select': 'DOI,type,title,relation',
          'rows': 3}

r = session.get(url, params=params)
r.json()['message']['items']

[{'DOI': '10.1364/opticaopen.29459153.v1',
  'type': 'posted-content',
  'title': ['Optical Frequency Comb Bandwidth Enhancement of a Monolithically Integrated Gain Switched Device'],
  'relation': {'is-preprint-of': [{'id-type': 'doi',
     'id': '10.1364/OE.572415',
     'asserted-by': 'subject'}]}},
 {'DOI': '10.1364/opticaopen.31435669',
  'type': 'posted-content',
  'title': ['Nonlinear Photonic Neural Network With Frequency Comb Lines'],
  'relation': {'is-preprint-of': [{'id-type': 'doi',
     'id': '10.1364/OL.596734',
     'asserted-by': 'subject'},
    {'id-type': 'doi', 'id': '10.1364/OL.596734', 'asserted-by': 'object'}]}},
 {'DOI': '10.21203/rs.3.rs-6498356/v1',
  'type': 'posted-content',
  'title': ['Surface plasmon enhanced linear and nonlinear optical properties of lycopene bioconjugated silver nanoparticles'],
  'relation': {'is-preprint-of': [{'id-type': 'doi',
     'id': '10.1038/s41598-025-13103-2',
     'asserted-by': 'subject'}]}}]

## 💶 Funders endpoint

Funders are identified by their **Funder ID** from the [Open Funder Registry](https://www.crossref.org/services/funder-registry/) - a DOI with the prefix `10.13039`, e.g. `10.13039/501100001711` for the Swiss National Science Foundation (SNSF). In the `/funders` endpoint, only the second part (`501100001711`) is used. Funders can be searched with `query` and filtered by `location`. (Please note that Crossref is gradually moving from Funder IDs to [ROR IDs](https://ror.org/) for funders.)

In [47]:
# Search for funders
params = {'mailto': email,
          'query': 'swiss national science foundation'}

r = session.get(base_url + 'funders', params=params)
[[i['id'], i['name'], i['location']] for i in r.json()['message']['items']]

[['501100001711',
  'Schweizerischer Nationalfonds zur Förderung der Wissenschaftlichen Forschung',
  'Switzerland']]

In [48]:
# How many funders are located in Switzerland?
params = {'mailto': email,
          'filter': 'location:Switzerland',
          'rows': 10}

r = session.get(base_url + 'funders', params=params)
data = r.json()
print('total results =', data['message']['total-results'])
[i['name'] for i in data['message']['items']]

total results = 553


['GAVI Alliance',
 'Novartis',
 'Roche',
 'Novartis Foundation for Sustainable Development',
 'Global Fund to Fight AIDS, Tuberculosis and Malaria',
 'World Health Organization',
 'International Union Against Cancer',
 'UBS',
 'Rolex Institute',
 'Actelion Pharmaceuticals']

In [49]:
# A single funder: names, hierarchy and number of funded works
r = session.get(base_url + 'funders/501100001711', params={'mailto': email})
funder = r.json()['message']

print(list(funder))
print(funder['name'])
print(funder['alt-names'])
print(funder['work-count'], funder['descendant-work-count'])

['hierarchy-names', 'replaced-by', 'work-count', 'name', 'descendants', 'descendant-work-count', 'id', 'tokens', 'replaces', 'uri', 'hierarchy', 'alt-names', 'location']
Schweizerischer Nationalfonds zur Förderung der Wissenschaftlichen Forschung
['Fondo Nazionale Svizzero', 'Fonds National Suisse', 'Schweizerische Nationalfonds', 'Fondo Nazionale Svizzero per la Ricerca Scientifica', 'SNSF', 'Fonds National Suisse de la Recherche Scientifique', 'SNF', 'Swiss National Science Foundation', 'FNS', 'The Swiss National Science Foundation (SNSF)', 'Schweizerischer Nationalfonds']
87601 88506


In [50]:
# Sub-units of the funder (e.g. National Centres of Competence in Research)
list(funder['hierarchy-names'].values())

['Schweizerischer Nationalfonds zur Förderung der Wissenschaftlichen Forschung',
 'NCCR Catalysis',
 'National Center of Competence in Research Materials’ Revolution: Computational Design and Discovery of Novel Materials',
 'Swiss National Centre of Competence in Research Kidney Control of Homeostasis',
 'National Center of Competence in Research Quantum Science and Technology',
 'National Center of Competence in Research Affective Sciences - Emotions in Individual Behaviour and Social Processes',
 'nccr – on the move']

In [51]:
# Works funded by the SNSF: published per year and by type, using facets
params = {'mailto': email,
          'facet': 'published:*,type-name:*',
          'rows': 0}

r = session.get(base_url + 'funders/501100001711/works', params=params)
facets = r.json()['message']['facets']
print(sorted(facets['published']['values'].items(), reverse=True)[:10])
facets['type-name']['values']

[('2027', 74), ('2026', 7540), ('2025', 8146), ('2024', 7434), ('2023', 7367), ('2022', 7651), ('2021', 8310), ('2020', 7727), ('2019', 7124), ('2018', 6808)]


{'Journal Article': 84483,
 'Posted Content': 2211,
 'Chapter': 780,
 'Conference Paper': 733,
 'Edited Book': 98,
 'Dataset': 85,
 'Monograph': 49,
 'Book': 46,
 'Report': 13,
 'Other': 6,
 'Reference': 1,
 'Book Series': 1}

## 🏢 Members endpoint

**Members** are the organizations registering DOIs with Crossref - mostly publishers, but also universities, libraries, societies and funders. Each member has a numeric ID, one or more DOI prefixes, and statistics about its registered content. Especially interesting: the `coverage` information, telling you **how complete the deposited metadata** of a member is (share of records with abstracts, references, ORCIDs, licenses, funders, affiliations ...).

In [52]:
# Search for members
params = {'mailto': email,
          'query': 'university of bern'}

r = session.get(base_url + 'members', params=params)
[[i['id'], i['primary-name'], i['counts']['total-dois']] for i in r.json()['message']['items']]

[[22676, 'University of Bern', 8614]]

In [53]:
# A single member: Wiley (member ID 311)
r = session.get(base_url + 'members/311', params={'mailto': email})
member = r.json()['message']

print(list(member))
print(member['primary-name'], '-', member['location'])
print(member['counts'])
print(len(member['prefixes']), 'prefixes, e.g.', member['prefixes'][:5])

['last-status-check-time', 'primary-name', 'counts', 'breakdowns', 'prefixes', 'coverage', 'prefix', 'id', 'tokens', 'counts-type', 'coverage-type', 'flags', 'location', 'names']
Wiley - Hoboken, NJ, United States
{'current-dois': 1197272, 'backfile-dois': 11051490, 'total-dois': 12248762}
60 prefixes, e.g. ['10.1506', '10.1901', '10.3401', '10.1301', '10.1348']


The `coverage` attributes are given separately for **current** content (published in the current and the two previous years) and **backfile** content (older). Let's compare the metadata coverage of two members in a dataframe.

In [54]:
coverage = {}
for member_id in [311, 22676]:     # Wiley and University of Bern
    r = session.get(base_url + f'members/{member_id}', params={'mailto': email})
    m = r.json()['message']
    coverage[m['primary-name']] = {k.replace('-current', ''): round(v, 2) for k, v in m['coverage'].items()
                                   if k.endswith('-current')}
    time.sleep(0.5)

pd.DataFrame(coverage)

,Wiley,University of Bern
affiliations,0.68,0.01
similarity-checking,0.84,0.66
descriptions,0.00,0.00
ror-ids,0.00,0.00
funders,0.30,0.00
funder-ror-ids,0.00,0.00
update-policies,0.73,0.00
orcids,0.63,0.26
licenses,0.78,0.80
award-numbers,0.26,0.00


In [55]:
# Works of a member can be accessed via /members/{id}/works - here: the content types of the University of Bern's DOIs
params = {'mailto': email,
          'facet': 'type-name:*',
          'rows': 0}

r = session.get(base_url + 'members/22676/works', params=params)
data = r.json()
print('total results =', data['message']['total-results'])
data['message']['facets']['type-name']['values']

total results = 8624


{'Journal Article': 7665,
 'Chapter': 683,
 'Journal Issue': 259,
 'Journal': 10,
 'Monograph': 2,
 'Edited Book': 2,
 'Posted Content': 1,
 'Peer Review': 1,
 'Other': 1}

## 📰 Journals endpoint

Journals are identified by their [ISSN](https://www.issn.org/) (print or electronic). The `/journals` endpoint provides information on the publisher, subjects, the number of DOIs per year and - again - the metadata coverage. Journals can be searched by title with `query`.

In [56]:
# Search for journals
params = {'mailto': email,
          'query': 'digital humanities',
          'rows': 10}

r = session.get(base_url + 'journals', params=params)
[[i['title'], i['publisher'], i['ISSN']] for i in r.json()['message']['items']]

[['DIGITAL HUMANITIES RESEARCH',
  'Institute of Russian Literature Pushkinskij Dom RAN',
  ['3034-4522']],
 ['Digital Humanities Quarterly',
  'The Association for Computers and the Humanities',
  ['1938-4122']],
 ['Digital Humanities Intersections',
  'Jaya Prakash Narayan National Centre of Excellence in the Humanities, IIT Indore',
  []],
 ['Frontiers in Digital Humanities', 'Frontiers Media SA', ['2297-2668']],
 ['Reviews in Digital Humanities',
  'PubPub (Knowledge Futures)',
  ['2766-9297']],
 ['SOCIAL SCIENCES & DIGITAL HUMANITIES',
  'D. Serikbayev East Kazakhstan Technical University',
  ['3107-2755', '3107-2763']],
 ['Digital Islamicate and Humanities',
  'PT Azki Inovasi Nusantara',
  ['3124-9523']],
 ['Digital Scholarship in the Humanities',
  'Oxford University Press',
  ['2055-7671', '2055-768X']],
 ['Infotheca - Journal for Digital Humanities',
  'Faculty of Philology, University of Belgrade',
  ['1450-9687', '2217-9461']],
 ['International Journal of Digital Humanities

In [57]:
# A single journal: Nature
r = session.get(base_url + 'journals/0028-0836', params={'mailto': email})
journal = r.json()['message']

print(list(journal))
print(journal['title'], '-', journal['publisher'])
print(journal['issn-type'])
print(journal['counts'])

['last-status-check-time', 'counts', 'breakdowns', 'publisher', 'coverage', 'title', 'subjects', 'coverage-type', 'flags', 'ISSN', 'issn-type']
Nature - Springer Science and Business Media LLC
[{'type': 'print', 'value': '0028-0836'}, {'type': 'electronic', 'value': '1476-4687'}]
{'current-dois': 12784, 'backfile-dois': 435112, 'total-dois': 447896}


In [58]:
# Metadata coverage of the journal's current content
journal['coverage-type']['current']

{'last-status-check-time': 1793813637442,
 'affiliations': 0.0,
 'abstracts': 0.1174906132665832,
 'orcids': 0.3918178973717146,
 'affiliation-ror-ids': 0.0,
 'licenses': 1.0,
 'references': 0.6386107634543179,
 'funders': 0.0,
 'similarity-checking': 1.0,
 'funder-ror-ids': 0.0,
 'award-numbers': 0.0,
 'ror-ids': 0.0,
 'update-policies': 0.3391739674593242,
 'resource-links': 1.0,
 'descriptions': 0.0}

In [59]:
# Number of DOIs per publication year as dataframe
df_years = pd.DataFrame(journal['breakdowns']['dois-by-issued-year'], columns=['year', 'dois']).sort_values('year')
df_years.tail(10)

,year,dois
41,2017,3754
30,2018,3938
28,2019,3981
51,2020,3622
44,2021,3737
24,2022,4103
18,2023,4320
16,2024,4387
14,2025,4550
35,2026,3847


In [60]:
# The 5 most cited Nature articles published since 2020
params = {'mailto': email,
          'filter': 'from-pub-date:2020-01-01,type:journal-article',
          'sort': 'is-referenced-by-count',
          'select': 'DOI,title,issued,is-referenced-by-count',
          'rows': 5}

r = session.get(base_url + 'journals/0028-0836/works', params=params)
[[i['is-referenced-by-count'], i['DOI'], i['title']] for i in r.json()['message']['items']]

[[47044,
  '10.1038/s41586-021-03819-2',
  ['Highly accurate protein structure prediction with AlphaFold']],
 [25911, '10.1038/s41586-020-2649-2', ['Array programming with NumPy']],
 [18209,
  '10.1038/s41586-024-07487-w',
  ['Accurate structure prediction of biomolecular interactions with AlphaFold 3']],
 [17580,
  '10.1038/s41586-020-2012-7',
  ['A pneumonia outbreak associated with a new coronavirus of probable bat origin']],
 [10089,
  '10.1038/s41586-020-2008-3',
  ['A new coronavirus associated with human respiratory disease in China']]]

## 🔢 Prefixes endpoint

Every DOI starts with a **prefix** (the part before the first `/`), which belongs to a Crossref member. The `/prefixes` endpoint tells you the owner of a prefix, and `/prefixes/{prefix}/works` gives access to all works registered with it.

In [61]:
r = session.get(base_url + 'prefixes/10.1063', params={'mailto': email})
r.json()['message']

{'member': 'https://id.crossref.org/member/317',
 'name': 'AIP Publishing',
 'prefix': 'https://id.crossref.org/prefix/10.1063'}

In [62]:
# Number of works registered with this prefix, by type
params = {'mailto': email,
          'facet': 'type-name:*',
          'rows': 0}

r = session.get(base_url + 'prefixes/10.1063/works', params=params)
data = r.json()
print('total results =', data['message']['total-results'])
data['message']['facets']['type-name']['values']

total results = 1025225


{'Journal Article': 728122,
 'Conference Paper': 285125,
 'Dataset': 7807,
 'Book': 2219,
 'Chapter': 1734,
 'Edited Book': 93,
 'Component': 52,
 'Report': 42,
 'Monograph': 24,
 'Other': 7}

## 🏷️ Types endpoint

The `/types` endpoint lists all **content types** used by Crossref. Their `id` is used in the `type` filter, e.g. `type:posted-content` for preprints. Note that OpenAlex uses its own list of types, so types are not directly comparable between both sources.

In [63]:
r = session.get(base_url + 'types', params={'mailto': email})
[[i['id'], i['label']] for i in r.json()['message']['items']]

[['book-section', 'Book Section'],
 ['monograph', 'Monograph'],
 ['report-component', 'Report Component'],
 ['report', 'Report'],
 ['peer-review', 'Peer Review'],
 ['book-track', 'Book Track'],
 ['journal-article', 'Journal Article'],
 ['book-part', 'Part'],
 ['other', 'Other'],
 ['book', 'Book'],
 ['journal-volume', 'Journal Volume'],
 ['book-set', 'Book Set'],
 ['reference-entry', 'Reference Entry'],
 ['proceedings-article', 'Proceedings Article'],
 ['journal', 'Journal'],
 ['component', 'Component'],
 ['book-chapter', 'Book Chapter'],
 ['proceedings-series', 'Proceedings Series'],
 ['report-series', 'Report Series'],
 ['proceedings', 'Proceedings'],
 ['database', 'Database'],
 ['standard', 'Standard'],
 ['reference-book', 'Reference Book'],
 ['posted-content', 'Posted Content'],
 ['journal-issue', 'Journal Issue'],
 ['dissertation', 'Dissertation'],
 ['grant', 'Grant'],
 ['dataset', 'Dataset'],
 ['book-series', 'Book Series'],
 ['edited-book', 'Edited Book']]

In [64]:
# Number of dissertations registered with Crossref, by publisher
params = {'mailto': email,
          'facet': 'publisher-name:10',
          'rows': 0}

r = session.get(base_url + 'types/dissertation/works', params=params)
data = r.json()
print('total results =', data['message']['total-results'])
data['message']['facets']['publisher-name']['values']

total results = 1101481


{"Agence Bibliographique de l'Enseignement Supérieur": 195213,
 'Universidade de São Paulo. Agência de Bibliotecas e Coleções Digitais': 130474,
 'Office of Academic Resources, Chulalongkorn University': 79083,
 'Universidade Estadual de Campinas': 67405,
 'National Documentation Centre (EKT)': 58554,
 'Iowa State University': 29896,
 'Faculdades Catolicas': 28317,
 'The University of Hong Kong Libraries': 27722,
 'Universidade de Sao Paulo, Agencia USP de Gestao da Informacao Academica (AGUIA)': 25588,
 'University of Queensland Library': 25270}

## 📜 Licenses endpoint

Many members deposit **license information** for their works - e.g. Creative Commons licenses for open access publications or text and data mining licenses. The `/licenses` endpoint lists the license URLs with the number of works using them. As the URLs are deposited by the members, you will find many variants of the same license (`http`/`https`, with or without `legalcode`, different spellings ...). So, when filtering works by `license.url`, consider the different variants.

In [65]:
params = {'mailto': email,
          'query': 'creativecommons.org/licenses/by/4.0',
          'rows': 10}

r = session.get(base_url + 'licenses', params=params)
data = r.json()
print('total results =', data['message']['total-results'])
data['message']['items']

total results = 135


[{'URL': 'http://creativecommons.org/LICENSEs/by/3.0/', 'work-count': 1},
 {'URL': 'http://creativecommons.org/license/by/3.0/', 'work-count': 5},
 {'URL': 'http://creativecommons.org/licenses/BY-SA/4.0', 'work-count': 1},
 {'URL': 'http://creativecommons.org/licenses/BY/4.0', 'work-count': 1},
 {'URL': 'http://creativecommons.org/licenses/by-nc-nd/3.0', 'work-count': 1},
 {'URL': 'http://creativecommons.org/licenses/by-nc-nd/3.0/',
  'work-count': 197},
 {'URL': 'http://creativecommons.org/licenses/by-nc-nd/4.0', 'work-count': 44},
 {'URL': 'http://creativecommons.org/licenses/by-nc-nd/4.0/',
  'work-count': 1972},
 {'URL': 'http://creativecommons.org/licenses/by-nc-sa/3.0/', 'work-count': 4},
 {'URL': 'http://creativecommons.org/licenses/by-nc-sa/4.0',
  'work-count': 574}]

In [66]:
# The most frequent licenses of works published in 2025, via facet
params = {'mailto': email,
          'filter': 'from-pub-date:2025-01-01,until-pub-date:2025-12-31',
          'facet': 'license:10',
          'rows': 0}

r = session.get(url, params=params)
r.json()['message']['facets']['license']['values']

{'https://www.elsevier.com/tdm/userlicense/1.0/': 1086930,
 'https://www.elsevier.com/legal/tdmrep-license': 1086903,
 'https://doi.org/10.15223/policy-037': 935032,
 'https://doi.org/10.15223/policy-029': 935032,
 'https://creativecommons.org/licenses/by/4.0/': 743145,
 'https://www.springernature.com/gp/researchers/text-and-data-mining': 591847,
 'https://creativecommons.org/licenses/by/4.0': 479575,
 'https://doi.org/10.15223/policy-017': 469777,
 'https://doi.org/10.15223/policy-012': 468229,
 'https://doi.org/10.15223/policy-004': 468229}

## ⚠️ Error handling and rate limits

Always check the **status code** of a response before processing it:
* `200`: success
* `400`: invalid request, e.g. a wrong filter or parameter name - the response explains the error
* `404`: the requested object does not exist (e.g. a DOI not registered with Crossref)
* `429`: too many requests - slow down (see the `Retry-After` header)
* `5xx`: server error - try again later, see the [status page](https://status.crossref.org/)

The error message of an invalid parameter is quite helpful, since it lists all valid values.

In [67]:
# An invalid filter name returns status 400 and a list of all valid filters
r = session.get(url, params={'mailto': email, 'filter': 'foo:bar', 'rows': 0})
print(r.status_code)
print(r.json()['message'][0]['message'][:600], '...')

400
Filter 'foo' specified but there is no such filter for this route. Valid filters for this route are: until-approved-date, has-assertion, from-print-pub-date, until-deposit-date, from-accepted-date, has-authenticated-orcid, gte-citation, from-created-date, relation.object, issn, ror-id, lte-award-amount, until-online-pub-date, group-title, full-text.application, has-funder-ror-id, until-created-date, license.version, from-deposit-date, has-abstract, from-awarded-date, has-event, from-approved-date, funder, assertion-group, from-online-pub-date, from-issued-date, directory, content-domain, licen ...


In [68]:
# The same works for facets, sort fields, field queries and select fields
r = session.get(url, params={'mailto': email, 'facet': 'foo:5', 'rows': 0})
print(r.json()['message'][0]['message'])

Facet 'foo' specified but there is no such facet for this route. Valid facets for this route are: *, affiliation, license, container-title, publisher-name, link-application, orcid, update-type, published, funder-name, archive, type-name, relation-type, ror-id, issn, source, journal-issue, category-name, assertion, assertion-group, journal-volume, funder-doi


In [69]:
# The current rate limits are given in the response headers - compare single records and lists
for path, p in [('works/10.1063/1.1699114', {}), ('works', {'query': 'test', 'rows': 0})]:
    r = session.get(base_url + path, params={'mailto': email, **p})
    print(path, {k: v for k, v in r.headers.items() if k.lower().startswith(('x-rate-limit', 'x-concurrency', 'x-api-pool'))})

works/10.1063/1.1699114 {'x-rate-limit-limit': '10', 'x-rate-limit-interval': '1s', 'x-concurrency-limit': '3', 'x-api-pool': 'polite-single'}


works {'x-rate-limit-limit': '3', 'x-rate-limit-interval': '1s', 'x-concurrency-limit': '3', 'x-api-pool': 'polite-array'}


The automatic retries of our session (see setup at the beginning) already handle `429` and server errors. If you do not use such a session, a small helper function does the same job: it pauses and retries if the server answers with `429` or a server error.

In [70]:
def get_crossref(path, params=None, retries=3):
    """Request the Crossref REST API, retrying on rate limit and server errors."""
    params = {'mailto': email, **(params or {})}
    for attempt in range(retries):
        r = session.get(base_url + path, params=params)
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(int(r.headers.get('Retry-After', 2 ** attempt)))
            continue
        r.raise_for_status()
        return r.json()['message']
    r.raise_for_status()

get_crossref('works', {'query.bibliographic': 'Einstein 1905 Zur Elektrodynamik bewegter Körper', 'rows': 1,
                       'select': 'DOI,title,score'})

{'facets': {},
 'total-results': 994311,
 'items': [{'DOI': '10.1002/andp.19053221004',
   'title': ['Zur Elektrodynamik bewegter Körper'],
   'score': 67.65016}],
 'items-per-page': 1,
 'query': {'start-index': 0, 'search-terms': None}}

## 🧰 Using a client library: Habanero

Instead of writing all requests yourself, you can use a client library. For Python, [Habanero](https://habanero.readthedocs.io/) is well maintained and covers all endpoints of the REST API. It handles parameters, cursors and errors for you. (The formerly used library [crossref-commons](https://gitlab.com/crossref/crossref_commons_py) is no longer maintained.)

    pip install habanero

In [71]:
from habanero import Crossref

# Set your e-mail address to use the polite pool, and a user agent string
cr = Crossref(mailto=email, ua_string='ds-pytools Crossref tutorial')
cr

< Crossref 
URL: https://api.crossref.org
KEY: None
MAILTO: your.name@example.org
ADDITIONAL UA STRING: ds-pytools Crossref tutorial
Timeout: 5
>

In [72]:
# Single work by DOI
cr.works(ids='10.1063/1.1699114')['message']['title']

['Equation of State Calculations by Fast Computing Machines']

In [73]:
# Query with filter, sorting, selection and limit - the parameters are the same as in the REST API
res = cr.works(query='gauge field theory', filter={'type': 'journal-article', 'from_pub_date': '2020'},
               sort='is-referenced-by-count', select=['DOI', 'title', 'is-referenced-by-count'], limit=5)
[[i['is-referenced-by-count'], i['DOI'], i['title']] for i in res['message']['items']]

[[7832,
  '10.1021/acs.jcim.1c00203',
  ['AutoDock Vina 1.2.0: New Docking Methods, Expanded Force Field, and Python Bindings']],
 [6909,
  '10.1145/3592433',
  ['3D Gaussian Splatting for Real-Time Radiance Field Rendering']],
 [5188,
  '10.1016/j.cedpsych.2020.101860',
  ['Intrinsic and extrinsic motivation from a self-determination theory perspective: Definitions, theory, practices, and future directions']],
 [4197,
  '10.1016/s0140-6736(20)32661-1',
  ['Safety and efficacy of the ChAdOx1 nCoV-19 vaccine (AZD1222) against SARS-CoV-2: an interim analysis of four randomised controlled trials in Brazil, South Africa, and the UK']],
 [3572,
  '10.1021/acs.jctc.1c00645',
  ['gmx_MMPBSA: A New Tool to Perform End-State Free Energy Calculations with GROMACS']]]

In [74]:
# Deep paging with cursor: Habanero collects the pages for you (here: max. 300 items)
res = cr.works(filter={'ror_id': '02k7v4d05', 'from_pub_date': '2024'}, select=['DOI'],
               cursor='*', cursor_max=300, limit=100)
print(len(res), 'pages')
print(sum(len(page['message']['items']) for page in res), 'items')

3 pages
300 items


In [75]:
# Other endpoints: funders, members, journals, prefixes, types, licenses
print(cr.funders(query='swiss national science foundation')['message']['items'][0]['name'])
print(cr.members(ids=311)['message']['primary-name'])
print(cr.journals(ids='0028-0836')['message']['title'])
print(cr.prefixes(ids='10.1063')['message']['name'])

Schweizerischer Nationalfonds zur Förderung der Wissenschaftlichen Forschung


Wiley


Nature


AIP Publishing


In [76]:
# Works of a funder, member, journal etc. with works=True
res = cr.funders(ids='501100001711', works=True, limit=3, select=['DOI', 'title'])
res['message']['items']

[{'DOI': '10.1386/chor_00058_3',
  'title': ['The dancing body as a living archive']},
 {'DOI': '10.1103/physrevb.101.184107',
  'title': ['High spatial resolution studies of phase transitions within organic aperiodic crystals']},
 {'DOI': '10.1007/s10334-024-01194-3',
  'title': ['Fast bias-corrected conductivity mapping using stimulated echoes']}]

Habanero also provides content negotiation via `https://doi.org` (`from habanero import cn`):

    from habanero import cn
    cn.content_negotiation(ids='10.1063/1.1699114', format='bibentry')               # BibTeX
    cn.content_negotiation(ids='10.1063/1.1699114', format='text', style='apa')      # formatted citation

More examples are in the [Habanero documentation](https://habanero.readthedocs.io/en/latest/).